# Volume Kedua Lambung Katamaran dengan Integrasi Numerik

**Tujuan.** Menghitung volume kedua lambung perahu katamaran dari gambar teknik (tampak atas dan tampak samping) yang diberi grid dash-kosong.

**Ringkasan hasil.** Volume kedua lambung sekitar **12,66 m³** (±12.660 liter), atau sekitar **6,33 m³** per lambung.

**Alur pengerjaan**
1. Menentukan skala gambar dari pola dash-kosong.
2. Membaca lebar dan tinggi lambung dari gambar pada beberapa posisi (stasiun).
3. Menyusun model luas penampang $A(x)$.
4. Mengintegralkan $A(x)$ dengan metode trapesium dan Simpson 1/3.
5. Menguji konvergensi dan kewajaran hasil.
6. Mengalikan 2 untuk kedua lambung.

## 1. Menentukan skala gambar

Tugas menyatakan bahwa **satu dash-kosong = 5 + 5 cm = 10 cm**. Artinya, satu pola berulang pada garis putus-putus grid (satu garis hijau + satu celah) mewakili 10 cm.

Gambar PDF dirender pada resolusi 200 dpi, lalu panjang pola dash-kosong diukur langsung dari piksel (27 pola berurutan = 450 px):

$$\text{1 pola} = \frac{450}{27} = 16{,}67\ \text{px} \;\Rightarrow\; \text{1 px} = \frac{10\ \text{cm}}{16{,}67\ \text{px}} = 0{,}6\ \text{cm}$$

Ukuran satu kotak grid juga diukur dari jarak antar garis grid: **71,2 px** (selisih antar kotak hanya 70,5 sampai 71,5 px, jadi seragam). Maka:

$$\text{1 kotak} = 71{,}2\ \text{px} = \frac{71{,}2}{16{,}67} = 4{,}27\ \text{pola} = 42{,}7\ \text{cm}$$

**Mengapa jumlah dash per kotak berbeda-beda (4 atau 5)?** Karena 4,27 bukan bilangan bulat, pola dash tidak pas dengan batas kotak, sehingga ada kotak yang terpotong di tengah dash. Karena itu skala **tidak boleh** dihitung sebagai "4 dash per kotak" (40 cm, terlalu kecil) atau "5 dash per kotak" (50 cm, terlalu besar). Skala yang benar diambil dari periode dash-kosong (10 cm = 16,67 px), atau rata-ratanya 42,7 cm per kotak.

## 2. Membaca dimensi lambung dari gambar

Lambung dibaca pada beberapa **stasiun**, yaitu posisi $x$ sepanjang lambung (sumbu mendatar gambar). Untuk tiap stasiun dicatat:

- **Lebar** $w(x)$ dari **tampak atas**: jarak vertikal antara tepi atas dan tepi bawah garis luar lambung pada posisi $x$.
- **Tinggi** $h(x)$ dari **tampak samping**: jarak vertikal dari garis geladak (tepi atas lambung) sampai garis dasar pada posisi yang sama $x$.

Semua nilai awal diukur dalam piksel, lalu dikonversi ke cm dengan faktor 0,6 cm/px.

**Batas bagian yang dihitung.** Reserve buoyancy tidak dihitung, jadi lambung dihitung dari garis reserve buoyancy kiri ($x = 201$ px) sampai kanan ($x = 1346$ px). Panjangnya sekitar 1145 px = 687 cm.

**Posisi yang tidak dibaca.** Dua daerah sekitar $x = 300$ sampai $345$ px dan $x = 1085$ sampai $1135$ px tertutup balok melintang antar lambung, sehingga garis luar lambung tidak terbaca di sana. Di bagian tengah ($x = 501$ sampai $1001$ px) lebar dan tinggi hampir konstan (150 px dan 184 sampai 185 px), jadi tidak perlu banyak stasiun. Nilai pada posisi yang tidak dibaca diisi dengan **interpolasi linear** antar stasiun terdekat.

**Pembacaan bentuk lambung**
- Buritan (kiri): dasar lambung naik dan lebar mengecil, tinggi bertambah dari 134 px menjadi 185 px.
- Tengah: lebar 150 px (90 cm) dan tinggi 185 px (111 cm) konstan.
- Haluan (kanan): lebar menyempit dari 150 px menjadi 91 px, sedangkan tinggi hampir konstan.

## 3. Model luas penampang

Karena lantai kapal dianggap datar (tanpa keel), tampak depan lambung berupa **persegi panjang** pada setiap posisi $x$. Maka luas penampang:

$$A(x) = w(x)\times h(x)$$

Volume satu lambung adalah jumlah dari "irisan tipis" penampang sepanjang lambung:

$$V = \int_{x_0}^{x_1} A(x)\,dx = \int_{x_0}^{x_1} w(x)\,h(x)\,dx$$

Integral ini dihitung secara numerik karena $w(x)$ dan $h(x)$ hanya diketahui di titik-titik stasiun, bukan sebagai rumus.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid, simpson

# Skala
CM_PER_POLA = 10.0          # 1 dash-kosong = 5 + 5 cm
PX_PER_POLA = 16.667        # hasil pengukuran gambar (200 dpi)
CM_PER_PX   = CM_PER_POLA / PX_PER_POLA   # = 0.6 cm/px

# Stasiun hasil pembacaan gambar (piksel): posisi x, lebar (tampak atas), tinggi (tampak samping)
x_px = np.array([201, 251, 351, 401, 451, 501, 1001, 1051, 1151, 1201, 1251, 1301, 1346])
w_px = np.array([123, 133, 144, 147, 149, 150,  149,  147,  137,  129,  119,  106,   91])
h_px = np.array([134, 153, 181, 185, 185, 185,  184,  184,  184,  184,  184,  184,  184])

x_cm = x_px * CM_PER_PX
w_cm = w_px * CM_PER_PX
h_cm = h_px * CM_PER_PX

def luas(x):
    return np.interp(x, x_cm, w_cm) * np.interp(x, x_cm, h_cm)   # cm2

print(f"Panjang lambung yang dihitung: {x_cm[-1]-x_cm[0]:.0f} cm")
print(f"Lebar maksimum: {w_cm.max():.0f} cm, tinggi: {h_cm.max():.0f} cm")

## 4. Metode 1: Aturan trapesium

Aturan trapesium menganggap $A(x)$ berubah **linear** di antara dua stasiun berurutan, lalu menjumlahkan luas trapesium:

$$V \approx \sum_{i=1}^{n-1} \frac{A(x_i) + A(x_{i+1})}{2}\,(x_{i+1}-x_i)$$

Di sini dipakai langsung pada stasiun hasil pembacaan gambar. Karena $A(x)$ sebenarnya hasil kali dua fungsi linear (kurva kuadratik), trapesium sedikit menyimpang, tetapi galatnya kecil bila jarak stasiun rapat.

In [ ]:
V_trap = trapezoid(w_cm * h_cm, x_cm)
print(f"Trapesium (stasiun data): {V_trap:,.0f} cm3 = {V_trap/1e6:.3f} m3")

## 5. Metode 2: Simpson 1/3

Simpson 1/3 memakai **parabola** yang melewati tiga titik berurutan, sehingga lebih akurat untuk kurva yang melengkung:

$$V \approx \frac{h}{3}\Big[A_0 + 4A_1 + 2A_2 + 4A_3 + \dots + 4A_{n-1} + A_n\Big]$$

dengan $h$ jarak antar titik yang seragam dan $n$ **genap** (jumlah titik ganjil). $A(x)$ dari dua fungsi linear adalah kuadratik, sehingga Simpson memberi nilai eksak untuk tiap bagian yang mulus. Untuk itu, $A(x)$ dievaluasi pada grid halus 1001 titik yang seragam (hasil interpolasi stasiun).

In [ ]:
n = 1001                                  # ganjil -> jumlah subinterval genap
xs = np.linspace(x_cm[0], x_cm[-1], n)
V_simp = simpson(luas(xs), x=xs)
print(f"Simpson 1/3 (n={n}): {V_simp:,.0f} cm3 = {V_simp/1e6:.3f} m3")

## 6. Uji konvergensi

Hasil trapesium dihitung ulang dengan jumlah titik yang makin banyak. Bila metodenya benar, nilainya mendekati hasil Simpson dan selisihnya mengecil.

In [ ]:
for n in [5, 10, 50, 200, 1000]:
    xs_ = np.linspace(x_cm[0], x_cm[-1], n)
    V = trapezoid(luas(xs_), xs_)
    print(f"n = {n:5d}  ->  V = {V:,.0f} cm3   (selisih vs Simpson: {abs(V - V_simp):,.0f} cm3)")

Selisih terhadap Simpson menurun dari sekitar 222.000 cm³ (5 titik) menjadi hanya 5 cm³ (1000 titik). Ini menunjukkan hasil sudah **konvergen**: menambah titik lagi tidak mengubah angka secara berarti.

## 7. Pengecekan kewajaran

Sebagai batas atas, bayangkan lambung berupa balok penuh dengan panjang, lebar, dan tinggi maksimum. Volume lambung sebenarnya harus **lebih kecil** dari balok itu karena buritan naik dan haluan menyempit.

In [ ]:
V_balok = (x_cm[-1] - x_cm[0]) * w_cm.max() * h_cm.max()
print(f"Balok batas atas : {V_balok/1e6:.2f} m3")
print(f"Hasil Simpson    : {V_simp/1e6:.2f} m3")
print(f"Rasio hasil/balok: {V_simp/V_balok:.2f}")

Rasio sekitar 0,92 masuk akal: lambung hampir berbentuk balok di bagian tengah, dan hanya ujung-ujungnya yang mengecil.

## 8. Hasil akhir

Kedua lambung dianggap identik, sehingga volume total = 2 x volume satu lambung.

In [ ]:
V_total = 2 * V_simp
print(f"Satu lambung  : {V_simp/1e6:.2f} m3 = {V_simp/1000:,.0f} L")
print(f"Kedua lambung : {V_total/1e6:.2f} m3 = {V_total/1000:,.0f} L")

## 9. Grafik

Grafik kiri menunjukkan profil lebar dan tinggi sepanjang lambung. Grafik kanan menunjukkan luas penampang $A(x)$; luas daerah yang diarsir sama dengan volume satu lambung.

In [ ]:
xs = np.linspace(x_cm[0], x_cm[-1], 400)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(xs, np.interp(xs, x_cm, w_cm), label="lebar")
ax[0].plot(xs, np.interp(xs, x_cm, h_cm), label="tinggi")
ax[0].set_xlabel("x (cm)"); ax[0].set_ylabel("cm"); ax[0].legend(); ax[0].set_title("Profil lambung")
ax[1].fill_between(xs, luas(xs), alpha=0.4)
ax[1].set_xlabel("x (cm)"); ax[1].set_ylabel("A(x) (cm2)"); ax[1].set_title("Luas penampang (area = volume)")
plt.tight_layout(); plt.show()

## 10. Keterbatasan dan kesimpulan

**Keterbatasan**
- Pembacaan lebar dan tinggi dari gambar mengandung ketidakpastian beberapa persen (ketebalan garis, resolusi gambar).
- Lambung dianggap berpenampang persegi panjang dan berlantai datar sesuai soal; bentuk sebenarnya (ada lengkung dan keel) akan memberi volume sedikit berbeda.
- Volume reserve buoyancy tidak dihitung sesuai soal.
- Nilai di antara stasiun diisi interpolasi linear.

**Kesimpulan.** Volume satu lambung sekitar **6,33 m³ (6.328 liter)**, dan volume kedua lambung sekitar **12,66 m³ (12.656 liter)**. Metode trapesium dan Simpson memberi hasil yang berselisih kurang dari 0,03%, dan uji konvergensi menunjukkan hasil yang stabil.